In [1]:
import run_model_preT
import run_model
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
#from imp import reload
import Adage
from scipy.stats import hypergeom
import csv
from AdageHyperModelv2 import AdageHyperModelv2
import tensorflow as tf
import shap
from keras.models import Model, load_model


from keras.layers import Layer, Dense
import tensorflow as tf
import keras

#import tensorflow.keras.backend as K


from keras import ops



import argparse
import numpy as np
import csv
import pandas as pd

from tensorflow.keras import optimizers, regularizers, layers, initializers, models
from tensorflow.keras.layers import Input, Dense, Dropout
#from keras.models import Model, Sequential
#from tensorflow.keras import initializers
import TiedWeightsEncoder as tw
import Adage as ad
from AdageHyperModel import AdageHyperModel
from AdageHyperModelv2 import AdageHyperModelv2

#from autoencoders import DenseLayerAutoencoder

#from DenseTranspose import DenseTranspose

from keras.models import Model, load_model
import keras_tuner
import matplotlib.pyplot as plt
import os

tf.compat.v1.enable_eager_execution()
print(tf.executing_eagerly())

True
True


In [27]:
keras.saving.get_custom_objects().clear()
@keras.saving.register_keras_serializable() #package="dla"
class DenseTranspose(keras.layers.Layer):
    def __init__(self, dense,  activation=None, **kwargs):
        self.dense = dense
        #self.layer_size = layer_size
        self.activation = keras.activations.get(activation)
        super().__init__(**kwargs)
    
    def build(self, batch_input_shape):
        self.biases = self.add_weight(name="bias", initializer="zeros",
            shape=[self.dense.get_weights()[0].shape[0]]) # input_shape  self.dense.input.shape[-1]
        super().build(batch_input_shape)
    
    def call(self, inputs, training=None):
        #t_weights = self.dense.get_weights()
        z = tf.matmul(inputs, self.dense.kernel, transpose_b=True)
        return self.activation(z + self.biases)

    def get_config(self):
        config = super().get_config()
        config.update({
           # "layer_size": self.layer_size,
            "dense": self.dense,
            #"activation": self.activation,
        })
        return config

    @classmethod
    def from_config(cls, config):

       # config["layer_size"] = keras.layers.deserialize(config["layer_size"])
        config["dense"] = keras.layers.deserialize(config["dense"])
        #config["activation"] = keras.layers.deserialize(config["activation"])
        return cls(**config)

In [28]:
inputs = Input(shape=(1000,))
d = Dropout(0.1)(inputs)
dense_1 = keras.layers.Dense(100, activation="selu")
dense_2 = keras.layers.Dense(30, activation="selu")
tied_encoder = keras.models.Sequential([
	inputs,
	dense_1,
	dense_2
	])
print(tied_encoder.summary())


Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 100)            │       100,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 30)             │         3,030 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,130 (402.85 KB)

 Trainable params: 103,130 (402.85 KB)

 Non-trainable params: 0 (0.00 B)

None


In [29]:
tied_decoder = keras.models.Sequential([
	DenseTranspose(dense_2, activation="selu"),
	DenseTranspose(dense_1, activation="sigmoid")
	])
print(tied_decoder.summary())
tied_decoder.save('tied_decoder_test.keras')
del tied_decoder
tied_decoder = load_model('tied_decoder_test.keras')
print(tied_decoder.summary())

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_transpose_13              │ ?                      │   0 (unbuilt) │
│ (DenseTranspose)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_transpose_14              │ ?                      │   0 (unbuilt) │
│ (DenseTranspose)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,130 (402.85 KB)

 Trainable params: 103,130 (402.85 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_transpose_13              │ ?                      │   0 (unbuilt) │
│ (DenseTranspose)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_transpose_14              │ ?                      │   0 (unbuilt) │
│ (DenseTranspose)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,130 (402.85 KB)

 Trainable params: 103,130 (402.85 KB)

 Non-trainable params: 0 (0.00 B)

None


In [30]:
tied_ae = keras.models.Sequential([tied_encoder, tied_decoder])
print(tied_ae.summary())

tied_ae.save('tied_ae_test.keras')
del tied_ae
tied_ae = load_model('tied_ae_test.keras')
print(tied_ae.summary())

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_16 (Sequential)      │ (None, 30)             │       103,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_17 (Sequential)      │ (None, 1000)           │       104,230 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 207,360 (810.00 KB)

 Trainable params: 207,360 (810.00 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_16 (Sequential)      │ (None, 30)             │       103,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_17 (Sequential)      │ (None, 1000)           │       104,230 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 207,360 (810.00 KB)

 Trainable params: 207,360 (810.00 KB)

 Non-trainable params: 0 (0.00 B)

None


In [31]:
inputs = Input(shape=(1000,))
d = Dropout(0.1)(inputs)
dense_1 = keras.layers.Dense(100, activation="selu")
dense_2 = keras.layers.Dense(30, activation="selu")
autoencoder = keras.models.Sequential([
	inputs,
	dense_1,
	dense_2
	])
print(autoencoder.summary())

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 100)            │       100,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 30)             │         3,030 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,130 (402.85 KB)

 Trainable params: 103,130 (402.85 KB)

 Non-trainable params: 0 (0.00 B)

None


In [32]:
autoencoder.save('dense_autoencoder_test.keras')

del autoencoder

autoencoder = load_model('dense_autoencoder_test.keras')
print(autoencoder.summary())
config = keras.layers.serialize(autoencoder)
print(config)

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 100)            │       100,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 30)             │         3,030 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,130 (402.85 KB)

 Trainable params: 103,130 (402.85 KB)

 Non-trainable params: 0 (0.00 B)

None
{'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential_19', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': (None, 1000), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_17'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense_18', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 100, 'activation': 'selu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regular

In [33]:
def linked_as_dt(encoding_dim, gene_num, act, init,seed, kl1, kl2, preT=False, init_weights=[]):
	inputs = Input(shape=(gene_num,))
	d = Dropout(0.1)(inputs)
	dense_1 = keras.layers.Dense(100, activation="selu")
	dense_2 = keras.layers.Dense(30, activation="selu")
	tied_encoder = keras.models.Sequential()
	tied_encoder.add(inputs)
	tied_encoder.add(dense_1)
	tied_encoder.add(dense_2)

	tied_encoder.add(DenseTranspose(dense_2, activation="selu"))
	tied_encoder.add(DenseTranspose(dense_1, activation="selu"))

	#tied_ae = keras.models.Sequential([tied_encoder, tied_decoder])
	return(tied_encoder)

In [34]:
def prep_data(all_comp, seed):
	all_comp_np = all_comp.values.astype("float64")
	gene_num = np.size(all_comp_np, 0)
	x_train = all_comp_np.transpose()
	# set noise factor
	noise_factor = 0.1
	# add noise
	x_train_noisy = x_train + (noise_factor
		            * np.random.normal(loc=0.0,scale=1.0, size=x_train.shape))
	# limit to range 0-1
	x_train_noisy = np.clip(x_train_noisy, 0., 1.)

	return(x_train, x_train_noisy)

In [35]:
all_comp = pd.read_csv('../data_files/se_se16_epi_comp_log_counts_norm_01.csv', index_col=0)
	
# this is the size of our input
gene_num = np.size(all_comp, 0)


	# prepare data
x_train, x_train_noisy = prep_data(all_comp, seed=1)
autoencoder = linked_as_dt(encoding_dim=300, gene_num=gene_num, act="relu", init="glorot_uniform",
								seed=1, kl1=0, kl2=0)
print(autoencoder.summary())

autoencoder.save('dense_autoencoder_test.keras')

del autoencoder

autoencoder = load_model('dense_autoencoder_test.keras')
print(autoencoder.summary())
config = keras.layers.serialize(autoencoder)
print(config)
print(gene_num)

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 100)            │       226,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 30)             │         3,030 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_transpose_15              │ (None, 100)            │         3,130 │
│ (DenseTranspose)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_transpose_16              │ (None, 2266)           │       228,966 │
│ (DenseTranspose)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 232,096 (906.62 KB)

 Trainable params: 232,096 (906.62 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 100)            │       226,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 30)             │         3,030 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_transpose_15              │ (None, 100)            │         3,130 │
│ (DenseTranspose)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_transpose_16              │ (None, 2266)           │       228,966 │
│ (DenseTranspose)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 232,096 (906.62 KB)

 Trainable params: 232,096 (906.62 KB)

 Non-trainable params: 0 (0.00 B)

None
{'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential_20', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': (None, 2266), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_18'}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense_20', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 100, 'activation': 'selu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regular

In [36]:
optim = optimizers.SGD(learning_rate=0.01, momentum=.9) # lr=0.001, rho=0.95, epsilon=1e-07
autoencoder.compile(optimizer=optim, 
	loss=tf.keras.losses.MeanSquaredError())

history = autoencoder.fit(x=x_train, #_noisy_f
	y=x_train_noisy, #_f
	epochs=10,
	batch_size=10,
	#shuffle=True,
	validation_split = 0.1,
	#verbose=1,
	#validation_data=(np.array(x_train_noise_test),
	#				 np.array(x_train_test)),
	verbose=True
	)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.2071 - val_loss: 0.1941
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1920 - val_loss: 0.1867
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1844 - val_loss: 0.1782
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1750 - val_loss: 0.1662
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1619 - val_loss: 0.1495
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1441 - val_loss: 0.1272
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1202 - val_loss: 0.1001
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0922 - val_loss: 0.0719
Epoch 9/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0647 - val_loss: 0.0483
Epoch 10/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0427 - val_loss: 0.0323
